In [ ]:
# importing libs

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

RANDOM_STATE = 42

In [ ]:
# dataset loading

df = pd.read_csv("Yangon-House-Price-Dataset.csv")
print(df.shape)
df.head()

(967, 13)


,price,bedroom,bathroom,sq_ft,aircorn,pcode,tsp_en,tsp_mm,floor,overhead_tank,parquet,tiles,price_sqft
0,200,0,1,625,0,MMR013014,Dawbon,ဒေါပုံ,5.0,NaN,NaN,NaN,32000.000000
1,200,0,1,960,0,MMR013040,Hlaing,လှိုင်,5.0,False,False,True,20833.333333
2,200,0,1,625,0,MMR013014,Dawbon,ဒေါပုံ,6.0,NaN,NaN,NaN,32000.000000
3,200,0,1,625,0,MMR013014,Dawbon,ဒေါပုံ,6.0,False,False,True,32000.000000
4,200,1,1,660,1,MMR013009,Thingangyun,သင်္ဃန်းကျွန်း,4.0,False,False,False,30303.030303


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 967 entries, 0 to 966
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          967 non-null    int64  
 1   bedroom        967 non-null    int64  
 2   bathroom       967 non-null    int64  
 3   sq_ft          967 non-null    int64  
 4   aircorn        967 non-null    int64  
 5   pcode          967 non-null    object 
 6   tsp_en         967 non-null    object 
 7   tsp_mm         967 non-null    object 
 8   floor          901 non-null    float64
 9   overhead_tank  798 non-null    object 
 10  parquet        798 non-null    object 
 11  tiles          798 non-null    object 
 12  price_sqft     967 non-null    float64
dtypes: float64(2), int64(5), object(6)
memory usage: 98.3+ KB


In [ ]:
# null value count 
df.isnull().sum()

price              0
bedroom            0
bathroom           0
sq_ft              0
aircorn            0
pcode              0
tsp_en             0
tsp_mm             0
floor             66
overhead_tank    169
parquet          169
tiles            169
price_sqft         0
dtype: int64

In [ ]:
# specific null cols

FEATURES = ["tsp_en", "bedroom", "bathroom", "aircorn", "floor", "overhead_tank"]
TARGET = "price"

data = df[FEATURES + [TARGET]].copy()
data.isnull().sum()

tsp_en             0
bedroom            0
bathroom           0
aircorn            0
floor             66
overhead_tank    169
price              0
dtype: int64

In [ ]:
# filling null values 

data["overhead_tank"] = data["overhead_tank"].map(
    {True: 1, False: 0, "True": 1, "False": 0}
)
data["overhead_tank"] = data["overhead_tank"].fillna(0)

In [ ]:
# changing data types

data["tsp_en"] = data["tsp_en"].astype(str)

In [ ]:
# review

data.isnull().sum()

tsp_en            0
bedroom           0
bathroom          0
aircorn           0
floor            66
overhead_tank     0
price             0
dtype: int64

In [9]:
data.describe()

,bedroom,bathroom,aircorn,floor,overhead_tank,price
count,967.000000,967.000000,967.000000,901.000000,967.000000,967.000000
mean,0.508790,0.943123,0.173733,3.176471,0.109617,387.823164
std,0.765235,0.340308,0.499288,2.052895,0.312574,78.374564
min,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000
25%,0.000000,1.000000,0.000000,1.000000,0.000000,325.000000
50%,0.000000,1.000000,0.000000,3.000000,0.000000,400.000000
75%,1.000000,1.000000,0.000000,5.000000,0.000000,450.000000
max,3.000000,2.000000,4.000000,8.000000,1.000000,500.000000


In [ ]:
# X, Y separate

numeric_features = ["bedroom", "bathroom", "aircorn", "floor", "overhead_tank"]
categorical_features = ["tsp_en"]

X = data[FEATURES]
y = data[TARGET]

X.head()

,tsp_en,bedroom,bathroom,aircorn,floor,overhead_tank
0,Dawbon,0,1,0,5.0,0.0
1,Hlaing,0,1,0,5.0,0.0
2,Dawbon,0,1,0,6.0,0.0
3,Dawbon,0,1,0,6.0,0.0
4,Thingangyun,1,1,1,4.0,0.0


In [ ]:
# 80-20 rule

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

((773, 6), (194, 6))

In [ ]:
# creating pipeline , transformer implementation 

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [ ]:
# training three models

candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
    results[name] = scores.mean()
    print(f"{name}: CV R^2 = {scores.mean():.4f} (+/- {scores.std():.4f})")

best_name = max(results, key=results.get)
print(f"\nBest model: {best_name}")

LinearRegression: CV R^2 = 0.2142 (+/- 0.0868)
RandomForest: CV R^2 = 0.1195 (+/- 0.1535)
GradientBoosting: CV R^2 = 0.1909 (+/- 0.1106)

Best model: LinearRegression


In [14]:
best_model = candidates[best_name]
final_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", best_model)])
final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedroom', 'bathroom',
                                                   'aircorn', 'floor',
                                                   'overhead_tank']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['tsp_en'])])),
                ('model', LinearRegression())])

In [ ]:
# evaluation

y_pred = final_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R^2  : {r2:.4f}")

MAE  : 62.30
RMSE : 72.46
R^2  : 0.1248


In [16]:
final_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedroom', 'bathroom',
                                                   'aircorn', 'floor',
                                                   'overhead_tank']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['tsp_en'])])),
                ('model', LinearRegression())])

In [ ]:
# model saving

model_bundle = {
    "pipeline": final_pipeline,
    "features": FEATURES,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "valid_townships": sorted(data["tsp_en"].unique().tolist()),
    "model_name": best_name,
    "metrics": {"mae": mae, "rmse": rmse, "r2": r2},
    "target": TARGET,
}

with open("house_price_model.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Saved house_price_model.pkl")
print("Valid townships:", model_bundle["valid_townships"])

Saved house_price_model.pkl
Valid townships: ['Ahlone', 'Bahan', 'Botahtaung', 'Dagon Myothit (North)', 'Dagon Myothit (South)', 'Dawbon', 'Hlaing', 'Insein', 'Kamaryut', 'Kyauktada', 'Kyeemyindaing', 'Lanmadaw', 'Mayangone', 'Mingalartaungnyunt', 'North Okkalapa', 'Pazundaung', 'Sanchaung', 'South Okkalapa', 'Tamwe', 'Thaketa', 'Thingangyun', 'Yankin']


In [ ]:
# predict

with open("house_price_model.pkl", "rb") as f:
    loaded_bundle = pickle.load(f)

loaded_pipeline = loaded_bundle["pipeline"]

def predict_price(tsp_en, bedroom, bathroom, aircorn, floor, overhead_tank):
    if tsp_en not in loaded_bundle["valid_townships"]:
        raise ValueError(f"Unknown township '{tsp_en}'.")

    row = pd.DataFrame([{
        "tsp_en": tsp_en,
        "bedroom": bedroom,
        "bathroom": bathroom,
        "aircorn": aircorn,
        "floor": floor,
        "overhead_tank": 1 if overhead_tank else 0,
    }])
    return loaded_pipeline.predict(row)[0]

predict_price(
    tsp_en="Dawbon",
    bedroom=2,
    bathroom=1,
    aircorn=1,
    floor=3,
    overhead_tank=True,
)

np.float64(347.5418286308124)